# 05 — So sánh mô hình dự đoán thắng/thua

**Phạm vi: phần C.** Dự đoán xác suất đội A thắng **một ván sau khi chọn đủ đội hình**. Fit encoder, baseline và mô hình trên train (231 ván); chỉ dùng validation (28 ván/7 series) để so sánh. **Không dùng test để chọn mô hình hoặc ngưỡng dự đoán.**

Notebook gọi `compare_on_validation()` từ `src/liqily/prediction/train.py`, không viết lại logic huấn luyện. Chạy các cell theo thứ tự từ trên xuống với kernel đã cài package (`python -m pip install -e '.[dev]'`). Kết quả thí nghiệm đã ghi ở cuối notebook để có thể đọc mà không cần chạy lại.

## Phương pháp và chỉ số

- Encoder cố định **83 tướng và 3 patch từ train**, tạo 173 cột. Tướng/patch chưa thấy trong train được mã hóa bằng cột unknown, nhưng tác động của chúng chưa được học.
- Hai baseline: tỷ lệ thắng chung của đội A và tỷ lệ thắng theo phe; cả hai chỉ fit trên train.
- Bốn cấu hình định trước: Logistic Regression với `C=0.1`/`1.0` (`solver=liblinear`, `max_iter=2000`); Random Forest với `max_depth=3`/`5` (`n_estimators=200`, `min_samples_leaf=5`, `n_jobs=1`). Seed của mô hình là `random_state=42`.
- **Brier score là chỉ số chính** (thấp hơn tốt hơn); log loss cũng đánh giá xác suất. Accuracy và balanced accuracy dùng ngưỡng 0,5 để bổ sung góc nhìn phân loại. Chọn cấu hình có Brier validation thấp nhất trong bốn mô hình; nếu hòa, lần lượt so log loss và tên cấu hình.
- Vì validation rất nhỏ, bảng điểm chỉ là kết quả thực nghiệm trên một giải đấu, không chứng minh mô hình tổng quát tốt.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import sklearn
from IPython.display import Markdown, display

# Support launching Jupyter from the project root or the notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'liqily').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from the project root or a subdirectory')
sys.path.insert(0, str(project_root / 'src'))

from liqily.prediction.data import DEFAULT_DATASET, audit_dataset
from liqily.prediction.train import candidate_models, compare_on_validation

schema = json.loads((DEFAULT_DATASET / 'feature_schema.json').read_text(encoding='utf-8'))
print('Dataset version:', schema['dataset_version'])
print('Python version:', sys.version.split()[0])
print('NumPy version:', np.__version__)
print('scikit-learn version:', sklearn.__version__)
print('Candidate settings (random_state=42):')
for name, model in candidate_models().items():
    print(f'  {name}: {model}')

Dataset version: 035ce761e53a47dbfd18ca1d9fabe7ceb0ec79115271bccac303bed2a6c0195c
Python version: 3.12.12
NumPy version: 2.4.2
scikit-learn version: 1.8.0
Candidate settings (random_state=42):
  logistic_C_0.1: LogisticRegression(C=0.1, max_iter=2000, random_state=42, solver='liblinear')
  logistic_C_1: LogisticRegression(max_iter=2000, random_state=42, solver='liblinear')
  forest_depth_3: RandomForestClassifier(max_depth=3, min_samples_leaf=5, n_estimators=200,
                       n_jobs=1, random_state=42)
  forest_depth_5: RandomForestClassifier(max_depth=5, min_samples_leaf=5, n_estimators=200,
                       n_jobs=1, random_state=42)


In [2]:
games, issues = audit_dataset(DEFAULT_DATASET)
if issues:
    raise ValueError(f'Dataset audit failed with {len(issues)} issue(s)')
train, validation = games['train'], games['validation']
# Test is audited for dataset integrity but is never passed to model comparison.
comparison = compare_on_validation(train, validation)
assert len(comparison.results) == 6
assert len(comparison.encoder.feature_names) == 173
rows = sorted(comparison.results, key=lambda row: (row['brier'], row['log_loss'], row['name']))
table = ['| Method | Brier ↓ | Log loss ↓ | Accuracy ↑ | Balanced accuracy ↑ |',
         '| --- | ---: | ---: | ---: | ---: |']
for row in rows:
    table.append(f'| {row["name"]} | {row["brier"]:.4f} | {row["log_loss"]:.4f} | '
                 f'{row["accuracy"]:.4f} | {row["balanced_accuracy"]:.4f} |')
display(Markdown('\n'.join(table)))
print('Training games:', len(train))
print('Validation games:', len(validation))
print('Feature count:', len(comparison.encoder.feature_names))
print('Best trained model by validation Brier:', comparison.best_model_name)

| Method | Brier ↓ | Log loss ↓ | Accuracy ↑ | Balanced accuracy ↑ |
| --- | ---: | ---: | ---: | ---: |
| forest_depth_3 | 0.2441 | 0.6815 | 0.6071 | 0.5417 |
| forest_depth_5 | 0.2449 | 0.6832 | 0.6429 | 0.5833 |
| majority_baseline | 0.2452 | 0.6835 | 0.5714 | 0.5000 |
| side_baseline | 0.2485 | 0.6902 | 0.5714 | 0.5000 |
| logistic_C_0.1 | 0.2551 | 0.7058 | 0.5357 | 0.4792 |
| logistic_C_1 | 0.2950 | 0.8692 | 0.6071 | 0.5625 |

Training games: 231
Validation games: 28
Feature count: 173
Best trained model by validation Brier: forest_depth_3


## Kết quả đã ghi nhận

Lần chạy với dataset `035ce761e53a47dbfd18ca1d9fabe7ceb0ec79115271bccac303bed2a6c0195c`, Python 3.12.12, NumPy 2.5.4 và scikit-learn 1.9.1:

| Phương pháp | Brier ↓ | Log loss ↓ | Accuracy ↑ | Balanced accuracy ↑ |
| --- | ---: | ---: | ---: | ---: |
| forest_depth_3 | **0,2441** | **0,6815** | 0,6071 | 0,5417 |
| forest_depth_5 | 0,2449 | 0,6832 | **0,6429** | **0,5833** |
| majority_baseline | 0,2452 | 0,6835 | 0,5714 | 0,5000 |
| side_baseline | 0,2485 | 0,6902 | 0,5714 | 0,5000 |
| logistic_C_0.1 | 0,2551 | 0,7058 | 0,5357 | 0,4792 |
| logistic_C_1 | 0,2950 | 0,8692 | 0,6071 | 0,5625 |

Đây là **bản ghi kết quả validation** của phiên bản và cấu hình nêu trên; bảng trong cell code phía trên được tính lại khi chạy notebook. Không dùng bảng tĩnh này thay cho thực nghiệm nếu dataset hoặc dependency thay đổi.

In [3]:
scores = {row['name']: row for row in comparison.results}
winner = scores[comparison.best_model_name]
baseline = scores['majority_baseline']
brier_gain = baseline['brier'] - winner['brier']
unseen_hero_games = sum(
    bool(details['team_a_heroes'] or details['team_b_heroes'])
    for details in (comparison.encoder.unknown_categories(game) for game in validation)
)
print('Brier improvement over majority baseline:', f'{brier_gain:.4f}')
print('Validation games with unseen heroes:', unseen_hero_games)
print('Validation series:', len({game.series_id for game in validation}))

Brier improvement over majority baseline: 0.0011
Validation games with unseen heroes: 3
Validation series: 7


## Diễn giải và giới hạn

`forest_depth_3` có Brier validation thấp nhất trong bốn cấu hình đã thử: **0,2441**. So với majority baseline **0,2452**, mức giảm chỉ khoảng **0,0011**. `forest_depth_5` có accuracy/balanced accuracy cao hơn nhưng không thắng theo chỉ số chính đã định trước. Không nên khẳng định rừng ngẫu nhiên vượt baseline một cách chắc chắn dựa trên chênh lệch này.

Validation chỉ có **28 ván thuộc 7 series** và có **3 ván chứa tướng chưa thấy trong train**. Mô hình không được học tác động của tướng mới; các ván trong cùng series cũng không phải những quan sát độc lập hoàn toàn. Việc thử nhiều cấu hình trên tập nhỏ có thể khiến thứ hạng không ổn định.

**Bước tiếp theo — phần D:** chốt quy tắc lựa chọn/ngưỡng từ validation, phân tích tính nhất quán khi đổi hai đội và chỉ sau đó đánh giá trên test **một lần**. Notebook này không trình bày số đo test và không dùng test để quyết định mô hình.